<a href="https://colab.research.google.com/github/cydneychelangatsang/MYPREDICT2-APPA/blob/main/Data%20Science.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 62.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [2]:
import pandas as pd

In [ ]:
# Load CSV file
file_path = '/content/sample_data/data_science_job.csv'
# Try reading with a different encoding, like 'latin-1'
try:
    df = pd.read_csv(file_path, encoding='latin-1')
except UnicodeDecodeError:
    # If 'latin-1' doesn't work, try another common encoding, like 'cp1252'
    try:
        df = pd.read_csv(file_path, encoding='cp1252')
    except UnicodeDecodeError:
        # If other common encodings fail, you might need to investigate the file encoding
        print("Could not decode the file with common encodings. Please check the file's actual encoding.")
        raise # Re-raise the error if decoding fails

In [5]:
print(df.columns.tolist())


['Company', 'Job Title', 'Location', 'Job Type', 'Experience level', 'Salary', 'Requirment of the company ', 'Facilities']


In [6]:
# Clean column names
df.columns = df.columns.str.strip()

# Now this should work
text_chunks = (
    df[['Job Title', 'Requirment of the company']]
    .fillna('')
    .astype(str)
    .apply(lambda row: f"{row['Job Title']}: {row['Requirment of the company']}", axis=1)
    .tolist()
)


In [7]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [8]:
# Import libraries
import numpy as np
import faiss
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM

In [9]:
# Define the embedding function
#convert a text string into a numerical embedding
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        model_output = model(**inputs)
    return model_output.last_hidden_state.mean(dim=1).squeeze().numpy()

In [10]:
# Generate embeddings
print("Creating embeddings...")
embeddings = []
for chunk in text_chunks:
    emb = get_embedding(chunk)
    embeddings.append(emb)

Creating embeddings...


In [11]:
# Convert to numpy array
embedding_matrix = np.array(embeddings).astype('float32')

In [12]:
# Create FAISS index
embedding_dim = embedding_matrix.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(embedding_matrix)

print(f"FAISS index has {index.ntotal} vectors.")

FAISS index has 3198 vectors.


In [13]:
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        model_output = model(**inputs)
    return model_output.last_hidden_state.mean(dim=1).squeeze().numpy()


In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
#using the FLAN-T5 model

# Load model
qa_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
qa_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate_answer_flan(query, retrieved_chunks):
    context = "\n".join(retrieved_chunks)
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"

    inputs = qa_tokenizer(prompt, return_tensors="pt", truncation=True, padding=True)
    outputs = qa_model.generate(**inputs, max_new_tokens=200)

    return qa_tokenizer.decode(outputs[0], skip_special_tokens=True)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [15]:
# Retrieve top-k similar chunks
def retrieve_chunks(query, top_k=3):
    query_embedding = get_embedding(query).astype('float32').reshape(1, -1)
    distances, indices = index.search(query_embedding, top_k)
    return [text_chunks[i] for i in indices[0]]


In [ ]:
while True:
    user_query = input("\nYour question: ")
    if user_query.lower() in ['exit', 'quit']:
        print("Goodbye!")
        break
    retrieved = retrieve_chunks(user_query)
    print("\nTop relevant context:")
    for i, r in enumerate(retrieved, 1):
        print(f"{i}. {r}")

    answer = generate_answer_flan(user_query, retrieved)
    print("\nAnswer:")
    print(answer)
